In [1]:
%cd ../
%load_ext autoreload
%autoreload 2

/home/ced/bachelor/final


# Clustered Renewable Energy Asset Centroids in Brandenburg

This notebook visualizes the capacity-weighted centroids of wind and solar installations across Brandenburg. These centroids optimize Numerical Weather Prediction (NWP) API query locations while maintaining meteorological representation.

* **Wind Fleet**: Clustered via Haversine DBSCAN ($3.0 \text{ km}$ radius).
* **Utility Solar**: Clustered via Haversine DBSCAN ($2.0 \text{ km}$ radius).
* **Distributed Solar**: Grouped via a static geographic grid ($0.1^\circ$ resolution).

In [2]:
import pandas as pd
import folium
import geopandas as gpd
import numpy as np

# Load datasets
wind_df = pd.read_csv('./data/processed/wind_clusters.csv')
solar_df = pd.read_csv('./data/processed/solar_clusters.csv')
boundary_path = './data/external/brandenburg_boundary.geojson'

# Brandenburg Centroid
map_center = [52.35, 13.0]

## 1. Wind Fleet Clusters Map

Teal circles represent wind clusters. The radius of each circle corresponds to the square root of the cluster's capacity (MW) for visibility. Hover or click on a cluster to see detailed stats.

In [3]:
# Create the Wind map
m_wind = folium.Map(location=map_center, zoom_start=8, tiles='cartodbpositron')

# Add Brandenburg boundary outline
try:
    boundary = gpd.read_file(boundary_path)
    folium.GeoJson(
        boundary, 
        style_function=lambda x: {'fillColor': '#ffffff', 'color': '#333333', 'weight': 1.5, 'fillOpacity': 0.1}
    ).add_to(m_wind)
except Exception as e:
    print(f"Could not load boundary: {e}")

# Add wind clusters
for _, row in wind_df.iterrows():
    # Scale radius based on capacity
    radius = np.sqrt(row['total_capacity_mw']) * 1.5 + 2.0
    
    popup_text = f"""
    <b>Cluster ID:</b> {row['cluster_id']}<br>
    <b>Centroid:</b> ({row['centroid_lat']:.4f}, {row['centroid_lon']:.4f})<br>
    <b>Capacity:</b> {row['total_capacity_mw']:.2f} MW<br>
    <b>Asset Count:</b> {int(row['asset_count'])}
    """
    
    folium.CircleMarker(
        location=[row['centroid_lat'], row['centroid_lon']],
        radius=radius,
        color='#008080',
        fill=True,
        fill_color='#00cbcb',
        fill_opacity=0.7,
        popup=folium.Popup(popup_text, max_width=250),
        tooltip=f"{row['cluster_id']}: {row['total_capacity_mw']:.1f} MW"
    ).add_to(m_wind)

m_wind

## 2. Utility-Scale Solar Clusters Map

Orange circles represent large ground-mounted solar installations grouped with a strict $2.0 \text{ km}$ DBSCAN radius.

In [4]:
# Create the Utility Solar map
m_solar_utility = folium.Map(location=map_center, zoom_start=8, tiles='cartodbpositron')

# Add boundary
try:
    folium.GeoJson(
        boundary, 
        style_function=lambda x: {'fillColor': '#ffffff', 'color': '#333333', 'weight': 1.5, 'fillOpacity': 0.1}
    ).add_to(m_solar_utility)
except Exception as e:
    pass

# Filter for utility-scale solar
utility_df = solar_df[solar_df['track_type'] == 'utility']

# Add utility solar clusters
for _, row in utility_df.iterrows():
    radius = np.sqrt(row['total_capacity_mw']) * 1.2 + 2.0
    
    popup_text = f"""
    <b>Cluster ID:</b> {row['cluster_id']}<br>
    <b>Centroid:</b> ({row['centroid_lat']:.4f}, {row['centroid_lon']:.4f})<br>
    <b>Capacity:</b> {row['total_capacity_mw']:.2f} MW<br>
    <b>Asset Count:</b> {int(row['asset_count'])}
    """
    
    folium.CircleMarker(
        location=[row['centroid_lat'], row['centroid_lon']],
        radius=radius,
        color='#ff5722',
        fill=True,
        fill_color='#ff8a50',
        fill_opacity=0.7,
        popup=folium.Popup(popup_text, max_width=250),
        tooltip=f"{row['cluster_id']}: {row['total_capacity_mw']:.1f} MW"
    ).add_to(m_solar_utility)

m_solar_utility

## 3. Distributed/Rooftop Solar Grid Centroids Map

Purple circles represent rooftop installations aggregated onto the $0.1^\circ$ geographic grid. The centroids are the exact centers of the grid cells.

In [5]:
# Create the Distributed Solar map
m_solar_dist = folium.Map(location=map_center, zoom_start=8, tiles='cartodbpositron')

# Add boundary
try:
    folium.GeoJson(
        boundary, 
        style_function=lambda x: {'fillColor': '#ffffff', 'color': '#333333', 'weight': 1.5, 'fillOpacity': 0.1}
    ).add_to(m_solar_dist)
except Exception as e:
    pass

# Filter for distributed solar
dist_df = solar_df[solar_df['track_type'] == 'distributed']

# Add distributed solar cells
for _, row in dist_df.iterrows():
    radius = np.sqrt(row['total_capacity_mw']) * 0.8 + 2.0
    
    popup_text = f"""
    <b>Grid Cell Centroid:</b> ({row['centroid_lat']:.2f}, {row['centroid_lon']:.2f})<br>
    <b>Total Capacity:</b> {row['total_capacity_mw']:.2f} MW<br>
    <b>Asset Count:</b> {int(row['asset_count'])}
    """
    
    folium.CircleMarker(
        location=[row['centroid_lat'], row['centroid_lon']],
        radius=radius,
        color='#7b1fa2',
        fill=True,
        fill_color='#ba68c8',
        fill_opacity=0.6,
        popup=folium.Popup(popup_text, max_width=250),
        tooltip=f"Grid Cell: {row['total_capacity_mw']:.1f} MW"
    ).add_to(m_solar_dist)

m_solar_dist

## 4. Open-Meteo Query Coordinates Summary

The following summary displays the exact number of coordinate centroids at which Numerical Weather Prediction (NWP) data will need to be downloaded from the Open-Meteo API.

In [6]:
# Print the exact coordinate counts for API querying
num_wind_pts = len(wind_df)
num_solar_utility_pts = len(solar_df[solar_df['track_type'] == 'utility'])
num_solar_dist_pts = len(solar_df[solar_df['track_type'] == 'distributed'])
total_api_points = num_wind_pts + num_solar_utility_pts + num_solar_dist_pts

print("Open-Meteo Query Points Summary:")
print(f"  - Wind Centroids: {num_wind_pts} coordinates")
print(f"  - Utility-scale Solar Centroids: {num_solar_utility_pts} coordinates")
print(f"  - Distributed Solar Grid Centroids: {num_solar_dist_pts} coordinates")
print(f"  - Total unique weather query locations: {total_api_points} points")

Open-Meteo Query Points Summary:
  - Wind Centroids: 211 coordinates
  - Utility-scale Solar Centroids: 455 coordinates
  - Distributed Solar Grid Centroids: 427 coordinates
  - Total unique weather query locations: 1093 points
